# Ordered Logistic Regression Results for Adoption Predictors in Rangeland Management Practices: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via its Croissant schema URL.

**Title:** Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya

**URL:** [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Dataset Description: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, noting their `@id` values. This lets us know what data structures are available and which `@id`s to use when extracting data and fields.

In [ ]:
# Exploring available record sets and their fields using `@id`
print("Available record sets and their fields (by '@id'):")
record_sets = []
for record_set in dataset.record_sets():
    print(f"\nRecord set '@id': {record_set.id}")
    record_sets.append(record_set.id)
    print("Fields/Columns in this record set:")
    for field in record_set.fields:
        print(f"  - Field '@id': {field.id}  (column: '{field.column}')")


## 3. Data Extraction

Load data from each record set into a DataFrame for further analysis, referencing record set and field `@id`s.

In [ ]:
# Extract all record sets present in the dataset
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set '@id': {record_set_id} => shape: {df.shape}")

# Example: Show columns for one record set (use the first one)
if record_sets:
    example_record_set = record_sets[0]
    print(f"\nColumns in record set '@id': {example_record_set}")
    print(dataframes[example_record_set].columns.tolist())
    display(dataframes[example_record_set].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)

Let's apply basic filtering, normalization, and grouping. We'll need to select a numeric field and a grouping field from one of the loaded record sets, using their `@id` as references.

Replace the placeholder IDs in the cell below with valid column IDs found in your dataset (as output above) if necessary.

In [ ]:
# Example: let's select a record set and fields by '@id'
if record_sets:
    selected_record_set_id = record_sets[0]  # Use the first record set for demonstration
    df = dataframes[selected_record_set_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields in record set '@id': {selected_record_set_id} => {numeric_fields}")

    # Pick the first available numeric field for analysis
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use its '@id'
        threshold = df[numeric_field_id].mean() if len(df) > 0 else 0
        print(f"\nFiltering records where '{numeric_field_id}' > {threshold:.2f}")

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records: {filtered_df.shape[0]}")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize the field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a non-numeric field
        non_numeric_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field:
            print(f"\nGrouping by field '@id': {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable group field available.")
    else:
        print("No numeric field found in example record set.")
else:
    print("No record sets found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields. We'll use Matplotlib and Seaborn for plotting (install as needed).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization example: Histogram and Boxplot for the selected numeric field
if record_sets and numeric_fields:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of '{numeric_field_id}'")

    plt.tight_layout()
    plt.show()

    # If a grouping field is available, plot group means
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field_id])
        plt.title(f"Mean '{numeric_field_id}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Not enough numeric data available for visualization.")

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Access and load FAIR² dataset metadata and record sets using `mlcroissant`
- Explore dataset structure, record sets, and fields (by their `@id`)
- Load record sets into pandas DataFrames for programmatic analysis
- Conduct basic filtering, normalization, and group-based aggregation
- Visualize numeric data for deeper understanding

Use this template to further analyze or build models with the FAIR² dataset or other Croissant-backed data sources!